# Loss-function control: does KL-softmax also grok under InfoNCE's other hparams?

Single-cell follow-up. The InfoNCE pilot's `infonce_long_100k` grokked to 92% at 100k, but it changed six hparams from pilot v4 simultaneously (loss, M, lr, batch sizes, lambda, warmup). This notebook isolates the **loss function**: identical setup to `infonce_long_100k` — M=2, lr=1e-3, shard_batch=512, consistency_batch=32, lambda=5, warmup=0, 100k steps — but swaps `consistency_loss` from `infonce` to `kl_softmax`.

| cell | M | loss | lambda | warmup | lr | shard_bs | cons_bs | steps |
|---|---|---|---|---|---|---|---|---|
| `kl_infonce_hparams_100k` | 2 | **KL-softmax**, train-inputs-only | 5 | 0 | 1e-3 | 512 | 32 | 100k |

If this cell groks too, the loss function was irrelevant and grokking is driven by the other hparam changes (small-M co-training, high lr, large shard batch, zero warmup, high lambda). If it plateaus, InfoNCE's representation-space contrast is the critical ingredient.

Budget: ~1.5h on a T4 (M=2, single cell).

## 1. Clone the repo

Pinned to `7b74afff` (same commit as the InfoNCE pilot that produced the 92% grok). This ensures any difference vs `infonce_long_100k` comes only from the loss-function swap, not from code drift.

In [ ]:
import os, subprocess, sys

GROK_REPO = os.environ.get('GROK_REPO', 'https://github.com/yazankb/grok.git')
GROK_COMMIT = os.environ.get('GROK_COMMIT', '7b74afff925f543cba16000e46ac032ce0d5b0bb')
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.expanduser('~/grok_work')
os.makedirs(WORK, exist_ok=True)
REPO_DIR = os.path.join(WORK, 'grok')

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', GROK_REPO, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', GROK_COMMIT], check=True)
print('repo at:', REPO_DIR)
print('commit :', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())

## 2. Install dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mod', 'sympy'], check=True)
import torch, pytorch_lightning as pl
print('torch =', torch.__version__, '| lightning =', pl.__version__, '| cuda =', torch.cuda.is_available())

## 3. Run the loss-control sweep

Single cell: `kl_infonce_hparams_100k`. Copies the exact config of `infonce_long_100k` (which grokked to 92%) but swaps `consistency_loss` from `infonce` to `kl_softmax`. Everything else — M=2, lr=1e-3, shard_batch=512, consistency_batch=32, lambda=5, warmup=0, 100k steps, train_inputs_only — is held constant.

Expected wall-clock: ~1.5h on T4 (M=2 is roughly half the per-step cost of M=4).

In [ ]:
import json

STEPS = int(os.environ.get('STEPS', 25000))
SWEEP_NAME = os.environ.get('SWEEP_NAME', 'loss_control_kl')
LOGDIR = os.environ.get('GROK_LOGDIR', os.path.join(WORK, 'consistency_runs'))
GPU = '0' if torch.cuda.is_available() else '-1'

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + ':' + env.get('PYTHONPATH', '')

CELLS_JSON = json.dumps([
    {
        'name': 'kl_infonce_hparams_100k',
        'description': 'KL-softmax with InfoNCE pilot hparams: M=2, lr=1e-3, shard_bs=512, cons_bs=32, lambda=5, warmup=0, 100k.',
        'random_seed': 42,
        'consistency_loss': 'kl_softmax',
        'consistency_lambda': 5,
        'consistency_warmup_steps': 0,
        'consistency_domain': 'train_inputs_only',
        'consistency_steps': 100000,
    },
])

cells_json_path = os.path.join(WORK, 'sweep_cells_loss_control.json')
with open(cells_json_path, 'w') as f:
    f.write(CELLS_JSON)

cmd = [
    sys.executable,
    os.path.join(REPO_DIR, 'scripts', 'run_consistency_sweep.py'),
    '--sweep_name', SWEEP_NAME,
    '--logdir', LOGDIR,
    '--consistency_steps', str(STEPS),
    '--gpu', GPU,
    '--n_models', '2',
    '--sharding', 'disjoint',
    '--train_data_pct', '50',
    '--max_lr', '1e-3',
    '--weight_decay', '0.1',
    '--shard_batch_size', '512',
    '--consistency_batch_size', '32',
    '--eval_every', '500',
    '--checkpoint_every', '0',
    '--log_every', '50',
    '--cells_json', cells_json_path,
]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env, cwd=REPO_DIR)

## 4. Inspect results

In [ ]:
import json

summary_path = os.path.join(LOGDIR, SWEEP_NAME, 'sweep_summary.json')
with open(summary_path) as f:
    summary = json.load(f)

print(f"sweep: {summary['sweep_name']}  ({len(summary['cells'])} cell(s))")
print(f"started: {summary.get('started_at')}  finished: {summary.get('finished_at')}")
print()
fmt = '{:<22} {:<8} {:>10} {:>12} {:>10}'
print(fmt.format('cell', 'status', 'best_ens', 'best_merged', 'sec'))
print('-' * 64)
for c in summary['cells']:
    r = c.get('results', {}) or {}
    print(fmt.format(
        c['name'], c['status'],
        f"{r.get('best_val_acc_ensemble', float('nan')):.2f}" if 'best_val_acc_ensemble' in r else 'n/a',
        f"{r.get('best_val_acc_merged', float('nan')):.2f}" if 'best_val_acc_merged' in r else 'n/a',
        f"{c.get('elapsed_sec', 0):.0f}",
    ))

## 5. Plot per-cell training curves

In [ ]:
import csv
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
(ax_ens, ax_merged), (ax_kl, ax_ent) = axes

for c in summary['cells']:
    if c['status'] != 'ok':
        continue
    eval_csv = os.path.join(c['experiment_dir'], 'consistency_eval.csv')
    if not os.path.isfile(eval_csv):
        continue
    with open(eval_csv) as f:
        rows = list(csv.DictReader(f))
    steps = [int(r['step']) for r in rows]
    ax_ens.plot(steps, [float(r['val_acc_ensemble']) for r in rows], label=c['name'])
    ax_merged.plot(steps, [float(r['val_acc_merged']) for r in rows], label=c['name'])
    ax_kl.plot(steps, [float(r['pairwise_kl_val_mean']) for r in rows], label=c['name'])
    ax_ent.plot(steps, [float(r['unsup_entropy_mean']) for r in rows], label=c['name'])

for ax, title in [
    (ax_ens, 'Ensemble val acc (%)'),
    (ax_merged, 'Merged val acc (%)'),
    (ax_kl, 'Pairwise val KL (alignment)'),
    (ax_ent, 'Unsup output entropy (collapse diag)'),
]:
    ax.set_title(title)
    ax.set_xlabel('step')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
ax_kl.set_yscale('log')
fig.tight_layout()
out_png = os.path.join(LOGDIR, SWEEP_NAME, 'sweep_curves.png')
fig.savefig(out_png, dpi=120)
print('saved:', out_png)
plt.show()

## 6. Bundle artifacts for download

In [ ]:
import shutil
sweep_dir = os.path.join(LOGDIR, SWEEP_NAME)
out_zip = os.path.join(WORK, f'{SWEEP_NAME}.zip')
if os.path.isfile(out_zip):
    os.remove(out_zip)
shutil.make_archive(out_zip[:-4], 'zip', root_dir=sweep_dir)
print('bundle:', out_zip, '|', os.path.getsize(out_zip) / 1e6, 'MB')